# Topic Modelling (NMF) — IEEE VIS Papers 1990–2024

This notebook applies **Non-negative Matrix Factorisation (NMF)** to the IEEE VIS publication corpus,
following the approach described in **Section 6.3** of the paper.

**Scenario:** The IEEE VIS dataset contains ~3,500 papers spanning 34 years of visualisation research.
We concatenate each paper's title and abstract into a single document, build a TF-IDF matrix
(top 300 unigrams + 200 bigrams), and apply NMF with varying numbers of topics (`n_topics = 5…25`)
to identify stable, interpretable research themes.

**Key differences from clustering notebooks:**
- **Algorithm:** NMF (matrix factorisation) instead of K-Means/DBSCAN
- **Features:** TF-IDF term weights (high-dimensional, sparse)
- **Iteration axis:** Varying `n_topics` (number of topics)
- **Interpretation:** Word clouds + term weights instead of spatial maps
- **Additional analysis:** Temporal prevalence of topics across publication years
- **Soft assignment:** Each document has a weight vector across all topics (not hard membership)

**Notebook structure:**

| Cell | Content |
|------|---------|
| 1 | Imports & configuration |
| 2 | Data loading & preprocessing |
| 3 | TF-IDF vectorisation (unigrams + bigrams) |
| 4 | NMF parameter sweep (compute all iterations) |
| 5 | Quality metrics across iterations |
| 6 | HDBSCAN archetype detection & colour assignment |
| 7 | Sankey flow diagram |
| 8 | Word clouds (topic profiles) |
| 9 | Interactive topic explorer |
| 10 | Transitions FROM a specific topic |
| 11 | Transitions TO a specific topic |
| 12 | Pair transition between two topics |
| 13 | Temporal prevalence analysis |
| 14 | Static HTML export |

### Cell 1: Imports and Configuration

All dependencies and global parameters are configured here. The key parameters follow
Section 6.3 of the paper:
- **n_topics range:** 5–25 (step 1) — sweeping the number of topics
- **Top unigrams:** 300 — most informative single terms
- **Top bigrams:** 200 — two-word phrases for richer semantics
- **NMF init:** 'nndsvda' — deterministic initialisation for reproducibility

In [ ]:
# ── Cell 1 — Imports and Configuration ───────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Patch
from matplotlib.gridspec import GridSpec
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from wordcloud import WordCloud, STOPWORDS
from collections import Counter, defaultdict
import re
import os
import math
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.decomposition import PCA
from sklearn.manifold import MDS
from scipy.ndimage import gaussian_filter1d

# ── Patch scipy for gensim compatibility ──
import scipy
import scipy.linalg
if not hasattr(scipy.linalg, 'triu'):
    scipy.linalg.triu = np.triu
if not hasattr(scipy.linalg, 'tril'):
    scipy.linalg.tril = np.tril

try:
    import gensim
    import gensim.corpora as corpora
    from gensim.models.coherencemodel import CoherenceModel
    HAS_GENSIM = True
    print(f'✅ gensim {gensim.__version__ if hasattr(gensim, "__version__") else ""} loaded (with scipy patch)')
except ImportError:
    HAS_GENSIM = False
    print('⚠ gensim not installed — coherence metrics will be skipped')
    print('  Install with: pip install gensim')

try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print('⚠ hdbscan not installed — archetype detection will be skipped')
    print('  Install with: pip install hdbscan')

# ── Configuration ──
DATA_DIR = 'data\\VisPapers'
CSV_FILE = os.path.join(DATA_DIR, 'IEEE VIS papers 1990-2024 - Main dataset.csv')

# NMF parameter sweep
NTOPICS_MIN = 5
NTOPICS_MAX = 25
NTOPICS_STEP = 1

# TF-IDF parameters
N_UNIGRAMS = 300       # top unigram terms
N_BIGRAMS = 200        # top bigram phrases
MIN_BIGRAM_DOCS = 3    # minimum document frequency for bigrams

# NMF parameters
NMF_INIT = None # 'nndsvda'   # deterministic initialisation
NMF_MAX_ITER = 400
RANDOM_STATE = 1

# Interpretation
N_TOP_WORDS = 15       # top words per topic for display
SIGMA_SMOOTH = 1.5     # Gaussian smoothing for temporal charts

SEED = RANDOM_STATE
#np.random.seed(SEED)

print('✅ Configuration loaded')
print(f'   Data:       {CSV_FILE}')
print(f'   Topics:     {NTOPICS_MIN}–{NTOPICS_MAX} (step {NTOPICS_STEP})')
print(f'   Vocabulary: {N_UNIGRAMS} unigrams + {N_BIGRAMS} bigrams')
print(f'   NMF init:   {NMF_INIT}, max_iter={NMF_MAX_ITER}')
print(f'   → {(NTOPICS_MAX - NTOPICS_MIN) // NTOPICS_STEP + 1} iterations planned')

### Cell 2: Load Data and Preprocess

We load the IEEE VIS papers dataset, concatenate Title + Abstract into a single
text document per paper, and clean encoding artefacts.

**Data cleaning (Iteration 0 from the paper):**
The paper describes discovering 283 papers containing `<<ETX>>` encoding artefacts
in their abstracts. These create a spurious stable topic with keywords like `etx`, `lt`, `gt`.
We remove these artefacts before analysis.

In [ ]:
# ── Cell 2 — Load and preprocess data ────────────────────────────────────────

# Load dataset
df = pd.read_csv(CSV_FILE)
print(f'Raw dataset: {len(df):,} papers')
print(f'Columns: {list(df.columns)}')
print(f'Year range: {df["Year"].min()} – {df["Year"].max()}')

# Keep only papers with both Title and Abstract
df = df.dropna(subset=['Title', 'Abstract']).reset_index(drop=True)
print(f'\nAfter dropping missing Title/Abstract: {len(df):,} papers')

# ── Data cleaning: remove <<ETX>> artefacts (Iteration 0 from paper) ──
etx_pattern = r'<<ETX>>|&lt;&lt;ETX&gt;&gt;|<ETX>'
etx_mask = df['Abstract'].str.contains(etx_pattern, regex=True, na=False)
n_etx = etx_mask.sum()
print(f'\n⚠ Papers with <<ETX>> artefact: {n_etx}')

# Remove the artefact strings
df['Abstract'] = df['Abstract'].str.replace(etx_pattern, '', regex=True)
df['Abstract'] = df['Abstract'].str.replace(r'\b(lt|gt|etx)\b', '', regex=True)
df['Abstract'] = df['Abstract'].str.strip()

# ── Concatenate Title + Abstract ──
df['text'] = df['Title'].fillna('') + ' ' + df['Abstract'].fillna('')
df['text'] = df['text'].str.strip()

# ═══════════════════════════════════════════════════════════════
# CRITICAL: Lowercase all text (simulates Java preprocessing)
# The standalone Python code expects pre-lowercased input.
# ═══════════════════════════════════════════════════════════════
df['text'] = df['text'].str.lower()

# Remove any papers with empty text after cleaning
df = df[df['text'].str.len() > 10].reset_index(drop=True)
print(f'Final corpus: {len(df):,} papers')

# Documents list for NMF
documents = df['text'].tolist()

# ── Summary statistics ──
print(f'\nYear distribution:')
print(f'  1990s: {((df["Year"] >= 1990) & (df["Year"] < 2000)).sum():,}')
print(f'  2000s: {((df["Year"] >= 2000) & (df["Year"] < 2010)).sum():,}')
print(f'  2010s: {((df["Year"] >= 2010) & (df["Year"] < 2020)).sum():,}')
print(f'  2020s: {(df["Year"] >= 2020).sum():,}')

# Show sample
display(df[['Year', 'Title', 'text']].head(3))

### Cell 3: TF-IDF Vectorisation

Following the paper's pipeline:
1. **Step 1:** Build top-N unigram vocabulary (with stop word removal)
2. **Step 2:** Find top-M bigrams from vocabulary terms (minimum document frequency ≥ 3)
3. **Step 3:** Inject bigrams into documents and build final TF-IDF matrix

The bigram detection is vocabulary-constrained: only pairs of terms that both
appear in the top-300 unigrams are considered as bigram candidates. This prevents
noise bigrams from diluting the vocabulary.

In [ ]:
# ── Cell 3 — TF-IDF vectorisation ────────────────────────────────────────────
#
# Matches TMQuality_compute.py exactly:
#   - lowercase=False (text already lowercased in Cell 2)
#   - stop_words='english' 
#   - Bigrams injected in-place into documents list
#   - Gensim dictionary built using vectorizer's analyzer (not manual split)

from datetime import datetime

tok_pattern = r'(?u)\b[a-zA-Z][\w]+\b'

# ── Step 1: Top-N unigrams ──
print(f'{datetime.now().strftime("%H:%M:%S")}. Step 1: Building top-{N_UNIGRAMS} unigram vocabulary...')

vec1 = TfidfVectorizer(
    lowercase=False,               # ← MATCHES STANDALONE (text already lowercase)
    token_pattern=tok_pattern,
    max_features=N_UNIGRAMS,
    stop_words='english'
)
vec1.fit(documents)
unigram_terms = set(vec1.get_feature_names_out())
print(f'  → {len(unigram_terms)} unigrams selected')

# ── Step 2: Find top bigrams from vocabulary terms ──
print(f'{datetime.now().strftime("%H:%M:%S")}. Step 2: Finding top-{N_BIGRAMS} bigrams...')

analyzer1 = vec1.build_analyzer()
bigram_counts = {}
bigram_doc_counts = {}

for doc in documents:
    tokens = analyzer1(doc)
    seen_in_doc = set()
    for j in range(len(tokens) - 1):
        w1, w2 = tokens[j], tokens[j + 1]
        if w1 == w2:  # skip self-bigrams
            continue
        if w1 in unigram_terms and w2 in unigram_terms:
            bg = w1 + '_' + w2
            bigram_counts[bg] = bigram_counts.get(bg, 0) + 1
            if bg not in seen_in_doc:
                bigram_doc_counts[bg] = bigram_doc_counts.get(bg, 0) + 1
                seen_in_doc.add(bg)

# Filter by minimum document frequency
min_bg_docs = max(MIN_BIGRAM_DOCS, len(documents) // 1000)
bigram_counts = {bg: c for bg, c in bigram_counts.items()
                 if bigram_doc_counts.get(bg, 0) >= min_bg_docs}

# Select top-M by frequency
top_bigrams = sorted(bigram_counts.keys(),
                     key=lambda x: bigram_counts[x], reverse=True)[:N_BIGRAMS]
print(f'  → {len(top_bigrams)} bigrams found (min doc freq={min_bg_docs})')
print(f'  Examples: {top_bigrams[:5]}')

# ── Inject bigrams INTO documents (in-place, like standalone) ──
if top_bigrams:
    bigram_set = set(top_bigrams)
    print(f'{datetime.now().strftime("%H:%M:%S")}. Injecting bigrams into documents...')
    
    for i in range(len(documents)):
        tokens = analyzer1(documents[i])
        merged = []
        j = 0
        while j < len(tokens):
            if j < len(tokens) - 1:
                candidate = tokens[j] + '_' + tokens[j + 1]
                if candidate in bigram_set:
                    merged.append(candidate)
                    j += 2
                    continue
            merged.append(tokens[j])
            j += 1
        documents[i] = ' '.join(merged)
    
    print(f'  Example bigram: {top_bigrams[0]}')

# Also keep a reference for word cloud generation
docs_processed = documents  # same list (modified in-place)

# ── Step 3: Final TF-IDF with combined vocabulary ──
print(f'{datetime.now().strftime("%H:%M:%S")}. Step 3: Building final TF-IDF matrix...')

explicit_vocab = sorted(unigram_terms) + list(top_bigrams)

vectorizer = TfidfVectorizer(
    lowercase=False,               # ← MATCHES STANDALONE
    token_pattern=tok_pattern,
    vocabulary=explicit_vocab,
    stop_words='english'
)
tfidf_matrix = vectorizer.fit_transform(documents)
terms = list(vectorizer.get_feature_names_out())

n_bigrams_in_vocab = sum(1 for t in terms if '_' in t)
print(f'  Final vocabulary: {len(terms)} terms '
      f'({len(terms) - n_bigrams_in_vocab} unigrams + {n_bigrams_in_vocab} bigrams)')
print(f'  TF-IDF matrix shape: {tfidf_matrix.shape}')

# Compute ||V||_F for reconstruction percentage
if hasattr(tfidf_matrix, 'data'):
    V_norm = float(np.sqrt(np.sum(tfidf_matrix.data ** 2)))
else:
    V_norm = float(np.linalg.norm(np.asarray(tfidf_matrix), 'fro'))
print(f'  ||V||_F = {V_norm:.4f}')

# ── Build Gensim dictionary for coherence ──
# CRITICAL: Use vectorizer's analyzer (same as standalone)
if HAS_GENSIM:
    print(f'{datetime.now().strftime("%H:%M:%S")}. Building Gensim dictionary...')
    
    analyzer_final = vectorizer.build_analyzer()
    tokenized_docs = [analyzer_final(doc) for doc in documents]
    
    id2word = corpora.Dictionary(tokenized_docs)
    
    # Diagnostic
    sample_words = terms[:10]
    found = sum(1 for w in sample_words if w in id2word.token2id)
    print(f'  Dictionary: {len(id2word)} tokens')
    print(f'  Diagnostic: {found}/{len(sample_words)} sample vocabulary terms found in dictionary')
    
    # Verify stop words are excluded
    from sklearn.feature_extraction._stop_words import ENGLISH_STOP_WORDS
    stop_in_vocab = [t for t in terms if t in ENGLISH_STOP_WORDS]
    print(f'  Stop words in vocabulary: {len(stop_in_vocab)} {stop_in_vocab[:5] if stop_in_vocab else "✓ none"}')
else:
    tokenized_docs = None
    id2word = None

print(f'\n✅ TF-IDF vectorisation complete')

In [ ]:
# ── Diagnostic: verify coherence setup ──
if HAS_GENSIM:
    # Check that topic-relevant words are in dictionary
    test_words = terms[:20]
    found_words = [w for w in test_words if w in id2word.token2id]
    missing_words = [w for w in test_words if w not in id2word.token2id]
    
    print(f"Vocabulary terms in gensim dictionary: {len(found_words)}/{len(test_words)}")
    if missing_words:
        print(f"  Missing: {missing_words[:10]}")
    
    # Check tokenized_docs aren't empty
    doc_lengths = [len(d) for d in tokenized_docs]
    print(f"Tokenized docs: {len(tokenized_docs)}, "
          f"mean length: {np.mean(doc_lengths):.1f}, "
          f"min: {min(doc_lengths)}, max: {max(doc_lengths)}")
    
    # Quick coherence test
    test_topics = [terms[:10]]  # one topic with first 10 vocab terms
    try:
        cm_test = CoherenceModel(
            topics=test_topics,
            texts=tokenized_docs,
            dictionary=id2word,
            coherence='c_v',
            processes=1
        )
        test_coh = cm_test.get_coherence()
        print(f"Test coherence (first 10 terms as one topic): {test_coh:.4f}")
    except Exception as e:
        print(f"Test coherence FAILED: {e}")

### Cell 4: NMF Parameter Sweep

Run NMF for each value of `n_topics` in the configured range. For each iteration:
- Fit NMF model → W (document-topic) and H (topic-term) matrices
- Normalise H rows (topic profiles) to sum to 1
- Compute quality metrics: reconstruction error, coherence, silhouette, diversity, exclusivity
- Assign each document to its dominant topic
- Store per-topic and per-document assignments

This mirrors the K-Means sweep over K, but the "clusters" are soft-assigned topics.

In [ ]:
# ── Cell 4 — NMF parameter sweep ─────────────────────────────────────────────

def topic_top_words(H_row, feature_names, n_top=10):
    """Get top-n words and weights for a topic."""
    top_idx = H_row.argsort()[-n_top:][::-1]
    return [feature_names[i] for i in top_idx], [float(H_row[i]) for i in top_idx]


def topic_exclusivity(H, topic_idx, n_top=10):
    """How exclusive are this topic's top words (not shared with other topics)."""
    H_row = H[topic_idx]
    top_idx = H_row.argsort()[-n_top:][::-1]
    scores = []
    for w_idx in top_idx:
        p_w_t = H_row[w_idx]
        p_w_others = max(H[t, w_idx] for t in range(H.shape[0]) if t != topic_idx)
        denom = p_w_t + p_w_others
        if denom > 1e-12:
            scores.append(p_w_t / denom)
    return float(np.mean(scores)) if scores else 0.0


# ── Run sweep ──
topic_values = list(range(NTOPICS_MIN, NTOPICS_MAX + 1, NTOPICS_STEP))
n_iterations = len(topic_values)

print(f'Running NMF: {n_iterations} iterations')
print(f'  n_topics: {topic_values[0]} → {topic_values[-1]} (step={NTOPICS_STEP})')
print(f'  Documents: {tfidf_matrix.shape[0]:,}')
print(f'  Terms: {tfidf_matrix.shape[1]:,}')
print('=' * 60)

# Storage
metrics_rows = []
topic_assignments = {}     # {n_topics: labels_array}
topic_weights = {}         # {n_topics: W_matrix}
topic_term_weights = {}    # {n_topics: H_matrix (normalised)}
top_words_per_iter = {}    # {n_topics: [[words], ...]}
iteration_params = {}      # {n_topics: {...}}

for iter_idx, n_topics in enumerate(topic_values):
    t0 = datetime.now()
    
    # Fit NMF
    model = NMF(
        n_components=n_topics,
        init=NMF_INIT,
        random_state=RANDOM_STATE,
        max_iter=NMF_MAX_ITER
    )
    W = model.fit_transform(tfidf_matrix)  # doc × topic
    H = model.components_.astype(float)     # topic × term
    
    # Reconstruction error
    frobenius = float(model.reconstruction_err_)
    reconstruction_pct = (1.0 - (frobenius**2 / V_norm**2)) * 100.0
    
    # Normalise H for interpretation
    H_norm = H / (H.sum(axis=1, keepdims=True) + 1e-12)
    
    # Document assignments (hard: argmax)
    labels = W.argmax(axis=1)
    
    # Top words per topic
    top_words_list = []
    for t in range(n_topics):
        words, weights_t = topic_top_words(H_norm[t], terms, N_TOP_WORDS)
        top_words_list.append(words)
    
    # Coherence (c_v) — with diagnostics and fallback
    if HAS_GENSIM and tokenized_docs is not None and id2word is not None:
        try:
            # Verify topic words exist in dictionary
            n_found = 0
            n_total = 0
            for tw in top_words_list:
                for w in tw:
                    n_total += 1
                    if w in id2word.token2id:
                        n_found += 1
            
            if n_found < n_total * 0.5:
                # Too many words missing — skip coherence
                if iter_idx == 0:
                    print(f'  ⚠ Only {n_found}/{n_total} topic words found in dictionary')
                avg_coherence = 0.0
                coherence_per_topic = [0.0] * n_topics
            else:
                cm = CoherenceModel(
                    topics=top_words_list,
                    texts=tokenized_docs,
                    dictionary=id2word,
                    coherence='c_v',
                    processes=1
                )
                avg_coherence = float(cm.get_coherence())
                coherence_per_topic = [float(c) for c in cm.get_coherence_per_topic()]
                
                # Handle NaN
                if np.isnan(avg_coherence):
                    avg_coherence = 0.0
                coherence_per_topic = [0.0 if np.isnan(c) else c for c in coherence_per_topic]
                
        except Exception as e:
            if iter_idx == 0:
                print(f'  ⚠ Coherence computation failed: {e}')
            avg_coherence = 0.0
            coherence_per_topic = [0.0] * n_topics
    else:
        avg_coherence = 0.0
        coherence_per_topic = [0.0] * n_topics
    
    # Silhouette (on W matrix)
    if n_topics >= 2:
        sil = silhouette_score(W, labels)
    else:
        sil = 0.0
    
    # Topic diversity (unique words across all topics)
    all_top_words = sum(top_words_list, [])
    global_diversity = len(set(all_top_words)) / len(all_top_words) if all_top_words else 0
    
    # Mean exclusivity
    exclusivities = [topic_exclusivity(H_norm, t, N_TOP_WORDS) for t in range(n_topics)]
    mean_exclusivity = float(np.mean(exclusivities))
    
    # Document counts per topic
    dominant_counts = [(labels == t).sum() for t in range(n_topics)]
    
    # Membership probability (confidence)
    W_sum = W.sum(axis=1, keepdims=True)
    W_prob = W / (W_sum + 1e-12)
    membership_probs = W_prob.max(axis=1)
    
    # Store results
    topic_assignments[n_topics] = labels.copy()
    topic_weights[n_topics] = W.copy()
    topic_term_weights[n_topics] = H_norm.copy()
    top_words_per_iter[n_topics] = top_words_list
    iteration_params[n_topics] = {
        'n_topics': n_topics,
        'n_documents': len(labels),
        'dominant_counts': dominant_counts,
        'coherence_per_topic': coherence_per_topic,
        'exclusivities': exclusivities,
        'membership_probs': membership_probs
    }
    
    metrics_rows.append({
        'Iteration': n_topics,
        'N_Topics': n_topics,
        'Frobenius': frobenius,
        'Reconstruction_Pct': reconstruction_pct,
        'Coherence_Cv': avg_coherence,
        'Silhouette': sil,
        'Global_Diversity': global_diversity,
        'Mean_Exclusivity': mean_exclusivity,
        'Mean_Membership_Prob': float(membership_probs.mean())
    })
    
    elapsed = (datetime.now() - t0).total_seconds()
    print(f'  n_topics={n_topics:>3}: '
          f'Recon={reconstruction_pct:.1f}%, Coh={avg_coherence:.3f}, '
          f'Sil={sil:.3f}, Div={global_diversity:.3f}, '
          f'Excl={mean_exclusivity:.3f}  [{elapsed:.1f}s]')

metrics_df = pd.DataFrame(metrics_rows)

ITER_MIN = topic_values[0]
ITER_MAX = topic_values[-1]

print(f'\n{"=" * 60}')
print(f'✅ Done: {n_iterations} iterations')
print(f'   Best coherence: {metrics_df["Coherence_Cv"].max():.4f} '
      f'at n_topics={metrics_df.loc[metrics_df["Coherence_Cv"].idxmax(), "N_Topics"]}')
print(f'   Best silhouette: {metrics_df["Silhouette"].max():.4f} '
      f'at n_topics={metrics_df.loc[metrics_df["Silhouette"].idxmax(), "N_Topics"]}')

### Cell 5: Quality Metrics Across Iterations

Visualise how topic quality changes as `n_topics` varies. Key metrics:
- **Reconstruction %** — how well NMF approximates the original TF-IDF matrix (↑ better)
- **Coherence (c_v)** — semantic coherence of top words within each topic (↑ better)
- **Silhouette** — separation quality based on document-topic weights (↑ better)
- **Global diversity** — fraction of unique words across all topics (↑ better)
- **Mean exclusivity** — how exclusive top words are to their topic (↑ better)
- **Mean membership probability** — confidence of document assignments (↑ better)

The paper finds coherence stabilising around n_topics = 14–18, suggesting ~15 topics.

In [ ]:
# ── Cell 5 — Quality metrics visualisation ───────────────────────────────────

fig, axes = plt.subplots(3, 2, figsize=(15, 12))

metric_configs = [
    ('Reconstruction_Pct', 'Reconstruction %', 'darkred', True),
    ('Coherence_Cv', 'Coherence (c_v)', 'green', True),
    ('Silhouette', 'Silhouette Score', 'blue', True),
    ('Global_Diversity', 'Global Diversity', 'purple', True),
    ('Mean_Exclusivity', 'Mean Topic Exclusivity', 'teal', True),
    ('Mean_Membership_Prob', 'Mean Membership Probability', 'orange', True),
]

for i, (col, title, color, higher_better) in enumerate(metric_configs):
    ax = axes.flat[i]
    ax.plot(metrics_df['N_Topics'], metrics_df[col], 'o-',
            color=color, markersize=5, linewidth=2)
    ax.set_xlabel('n_topics')
    ax.set_ylabel(col)
    ax.set_title(f'{title} ({"↑" if higher_better else "↓"} better)',
                 fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Mark best
    if higher_better:
        best_idx = metrics_df[col].idxmax()
    else:
        best_idx = metrics_df[col].idxmin()
    best_k = metrics_df.loc[best_idx, 'N_Topics']
    best_v = metrics_df.loc[best_idx, col]
    ax.axvline(best_k, color=color, linestyle='--', alpha=0.4)
    ax.annotate(f'best: {best_k}\n({best_v:.3f})',
                xy=(best_k, best_v), fontsize=8, color=color,
                textcoords='offset points', xytext=(10, -10))
    
    # Highlight n=15 region
    ax.axvspan(14, 18, alpha=0.05, color='green')

plt.suptitle(f'NMF Quality Metrics (n_topics={NTOPICS_MIN}–{NTOPICS_MAX})',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ── Summary table ──
display(HTML('<h4>Metrics Summary</h4>'))
display(metrics_df.style.format({
    'Frobenius': '{:.2f}', 'Reconstruction_Pct': '{:.1f}%',
    'Coherence_Cv': '{:.4f}', 'Silhouette': '{:.4f}',
    'Global_Diversity': '{:.4f}', 'Mean_Exclusivity': '{:.4f}',
    'Mean_Membership_Prob': '{:.4f}'
}).background_gradient(subset=['Coherence_Cv'], cmap='Greens')
 .background_gradient(subset=['Silhouette'], cmap='Blues'))

### Cell 6: HDBSCAN Archetype Detection & Colour Assignment

Following the paper's pipeline:
1. Stack all topic term-weight vectors (H rows) from all iterations into one matrix
2. Apply PCA + L2 normalisation for robust distance computation
3. Run HDBSCAN to detect recurring topic archetypes across iterations
4. Assign consistent colours via 2D embedding (MDS/UMAP)

Topics that recur across many n_topics values (same archetype) get the same colour.
Noise topics (HDBSCAN label = -1) are shown in grey.

In [ ]:
# ── Cell 6 — HDBSCAN archetype detection & colour assignment ──────────────────

# ── Stack all topic vectors ──
all_topic_vectors = []
topic_keys = []  # (n_topics, topic_id)

for n_topics in topic_values:
    H = topic_term_weights[n_topics]
    for t in range(n_topics):
        all_topic_vectors.append(H[t])
        topic_keys.append((n_topics, t))

X_all = np.array(all_topic_vectors)
print(f'Total topic instances: {X_all.shape[0]} (from {len(topic_values)} iterations)')

# ── Preprocessing: Standardise → PCA → L2 normalise ──
X_std = StandardScaler(with_mean=True, with_std=True).fit_transform(X_all)
n_pca = min(50, X_std.shape[0], X_std.shape[1])
X_pca = PCA(n_components=n_pca, random_state=0).fit_transform(X_std)
X_proc = Normalizer(norm='l2').fit_transform(X_pca)
print(f'After PCA({n_pca}) + L2 normalisation: {X_proc.shape}')

# ── HDBSCAN clustering ──
if HAS_HDBSCAN:
    # min_cluster_size ~ half the number of iterations
    min_cs = max(3, len(topic_values) // 2)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cs, metric='euclidean')
    archetype_labels = clusterer.fit_predict(X_proc)
    n_archetypes = len(set(archetype_labels)) - (1 if -1 in archetype_labels else 0)
    noise_frac = (archetype_labels == -1).sum() / len(archetype_labels)
    print(f'\nHDBSCAN: {n_archetypes} archetypes, {noise_frac:.1%} noise')
else:
    # Fallback: each topic in each iteration is its own "archetype"
    archetype_labels = np.arange(len(topic_keys))
    n_archetypes = len(topic_keys)
    noise_frac = 0.0
    print('\n⚠ HDBSCAN not available — using position-based colours')

# ── 2D embedding for colour assignment ──
print(f'Computing 2D embedding (MDS) for colour assignment...')
mds = MDS(n_components=2, random_state=0, dissimilarity='euclidean', normalized_stress='auto')
coords_2d = mds.fit_transform(X_proc)

# Normalise to [0, 1]
for dim in range(2):
    mn, mx = coords_2d[:, dim].min(), coords_2d[:, dim].max()
    if mx > mn:
        coords_2d[:, dim] = (coords_2d[:, dim] - mn) / (mx - mn)
    else:
        coords_2d[:, dim] = 0.5

# ── Colour function (four-corner interpolation from paper) ──
def get_color_from_position(x, y):
    """Four-corner colour interpolation."""
    reds = [228, 25, 255, 37]
    greens = [220, 228, 18, 13]
    blues = [0, 218, 6, 252]
    
    rr, cc = y, x
    rr1, cc1 = 1 - rr, 1 - cc
    dc = [math.sqrt(rr*rr + cc*cc), math.sqrt(rr*rr + cc1*cc1),
          math.sqrt(rr1*rr1 + cc*cc), math.sqrt(rr1*rr1 + cc1*cc1)]
    max_d = math.sqrt(2)
    weights = [max(0, (max_d - d) / max_d) for d in dc]
    
    dr = sum(w * reds[i] for i, w in enumerate(weights))
    dg = sum(w * greens[i] for i, w in enumerate(weights))
    db = sum(w * blues[i] for i, w in enumerate(weights))
    
    clamp = lambda c: max(0, min(255, int(c)))
    return f'#{clamp(dr):02x}{clamp(dg):02x}{clamp(db):02x}'

# ── Assign colours ──
NOISE_COLOR = '#808080'
topic_colors = {}  # (n_topics, topic_id) → hex_color

for i, (nt, tid) in enumerate(topic_keys):
    if archetype_labels[i] == -1:
        topic_colors[(nt, tid)] = NOISE_COLOR
    else:
        topic_colors[(nt, tid)] = get_color_from_position(
            coords_2d[i, 0], coords_2d[i, 1])

# Store colours per iteration
for n_topics in topic_values:
    colors = [topic_colors.get((n_topics, t), NOISE_COLOR) for t in range(n_topics)]
    iteration_params[n_topics]['colors'] = colors
    iteration_params[n_topics]['coords_2d'] = np.array([
        coords_2d[topic_keys.index((n_topics, t))] for t in range(n_topics)
    ])

print(f'\n✅ Colours assigned to {len(topic_colors)} topic instances')
print(f'   Archetypes: {n_archetypes}, Noise: {noise_frac:.1%}')

### Cell 7: Sankey Flow Diagram

Shows how document membership flows between topics as `n_topics` increases.
Each column = one iteration. Nodes = topics (sized by document count).
Bands = shared documents between consecutive iterations.

Key patterns:
- **Wide horizontal band** → stable topic persisting across iterations
- **Band splitting** → a topic differentiates into sub-themes
- **Thin bands from many sources** → wastebasket topic (like the "method" topic at K=16)

In [ ]:
# ── Cell 7 — Sankey flow diagram ─────────────────────────────────────────────

def build_topic_sankey(iter_range=None, max_display=12):
    """Build Sankey diagram for topic transitions."""
    if iter_range is None:
        iter_range = (ITER_MIN, ITER_MAX)
    
    iters = [k for k in topic_values if iter_range[0] <= k <= iter_range[1]]
    
    # Subsample if too many
    if len(iters) > max_display:
        step = len(iters) // max_display
        iters = iters[::step]
        if iters[-1] != topic_values[-1]:
            iters.append(topic_values[-1])
    
    # Build nodes
    node_labels, node_colors = [], []
    node_positions = {}  # (n_topics, topic_id) → node_index
    
    for it in iters:
        colors = iteration_params[it]['colors']
        counts = iteration_params[it]['dominant_counts']
        for t in range(it):
            idx = len(node_labels)
            node_positions[(it, t)] = idx
            # Top 3 words as label
            top3 = ', '.join(top_words_per_iter[it][t][:3])
            node_labels.append(f'K={it}.T{t}<br>{top3}<br>({counts[t]})')
            node_colors.append(colors[t])
    
    # Build links
    sources, targets, values, link_colors = [], [], [], []
    
    for i in range(len(iters) - 1):
        it1, it2 = iters[i], iters[i + 1]
        labels1 = topic_assignments[it1]
        labels2 = topic_assignments[it2]
        colors1 = iteration_params[it1]['colors']
        
        for t1 in range(it1):
            mask1 = (labels1 == t1)
            for t2 in range(it2):
                mask2 = (labels2 == t2)
                shared = int((mask1 & mask2).sum())
                if shared > 0:
                    sources.append(node_positions[(it1, t1)])
                    targets.append(node_positions[(it2, t2)])
                    values.append(shared)
                    c = colors1[t1]
                    r, g, b = int(c[1:3], 16), int(c[3:5], 16), int(c[5:7], 16)
                    link_colors.append(f'rgba({r},{g},{b},0.3)')
    
    fig = go.Figure(go.Sankey(
        node=dict(pad=12, thickness=18,
                  line=dict(color='black', width=0.5),
                  label=node_labels, color=node_colors),
        link=dict(source=sources, target=targets,
                  value=values, color=link_colors)
    ))
    fig.update_layout(
        title=f'Topic Flow: n_topics={iters[0]}→{iters[-1]}',
        height=700, width=1300, font_size=8)
    fig.show()


# Show full range
build_topic_sankey()

# Show detail around 15-16
print('\n── Detail: n_topics = 13–18 ──')
build_topic_sankey((13, 18), max_display=6)

### Cell 8: Word Clouds (Topic Profiles)

Word clouds are the primary interpretation tool for topic models (replacing
the Z-score profiles used for clustering). Two weighting modes:
- **Frequency-weighted:** Size ∝ term frequency within the topic's documents
- **TF-IDF/term-weight:** Size ∝ NMF weight in the H matrix (discriminative power)

The paper shows that 15 topics produce clearly interpretable, distinct research themes.

In [ ]:
# ── Cell 8 — Word clouds ──────────────────────────────────────────────────────

def show_word_clouds(n_topics, mode='term_weight', ncols=3):
    """Display word clouds for all topics at a given n_topics."""
    H = topic_term_weights[n_topics]
    colors = iteration_params[n_topics]['colors']
    counts = iteration_params[n_topics]['dominant_counts']
    labels = topic_assignments[n_topics]
    
    nrows = math.ceil(n_topics / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes_flat = axes.flatten() if n_topics > 1 else [axes]
    
    for t in range(n_topics):
        ax = axes_flat[t]
        color_hex = colors[t]
        rgb = tuple(int(color_hex[i:i+2], 16) for i in (1, 3, 5))
        
        if mode == 'term_weight':
            # Use NMF weights directly
            freq = {}
            for idx in H[t].argsort()[-60:][::-1]:
                if H[t, idx] > 1e-6:
                    freq[terms[idx]] = float(H[t, idx])
        else:
            # Frequency-weighted: count terms in topic's documents
            mask = (labels == t)
            topic_docs = [docs_processed[i] for i in range(len(docs_processed)) if mask[i]]
            word_counts = Counter()
            for doc in topic_docs:
                word_counts.update(doc.split())
            # Filter by vocabulary
            vocab_set = set(terms)
            freq = {w: c for w, c in word_counts.items()
                    if w in vocab_set and w.lower() not in STOPWORDS}
            # Keep top 60
            freq = dict(sorted(freq.items(), key=lambda x: -x[1])[:60])
        
        if not freq:
            ax.text(0.5, 0.5, f'T{t}\n(no terms)', ha='center', va='center',
                    transform=ax.transAxes, fontsize=12, color='gray')
            ax.axis('off')
            continue
        
        def color_func(word, **kwargs):
            return f'rgb({rgb[0]},{rgb[1]},{rgb[2]})'
        
        wc = WordCloud(
            width=600, height=400, max_words=60,
            background_color='white', color_func=color_func,
            prefer_horizontal=1.0, min_font_size=8,
            collocations=False
        ).generate_from_frequencies(freq)
        
        ax.imshow(wc, interpolation='bilinear')
        top3 = ', '.join(top_words_per_iter[n_topics][t][:3])
        ax.set_title(f'T{t}: {top3} (N={counts[t]})', fontsize=9, fontweight='bold')
        ax.axis('off')
    
    # Hide unused axes
    for idx in range(n_topics, len(axes_flat)):
        axes_flat[idx].axis('off')
    
    mode_label = 'TF-IDF weighted' if mode == 'term_weight' else 'Frequency weighted'
    fig.suptitle(f'Word Clouds — n_topics={n_topics} ({mode_label})',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# ── Widget ──
w_nt_wc = widgets.IntSlider(value=15, min=ITER_MIN, max=ITER_MAX, step=1,
                             description='n_topics:',
                             layout=widgets.Layout(width='300px'))
w_mode_wc = widgets.Dropdown(options=['term_weight', 'frequency'],
                              value='term_weight', description='Mode:')
btn_wc = widgets.Button(description='Show Word Clouds',
                         button_style='info')
out_wc = widgets.Output()

def on_btn_wc(b):
    with out_wc:
        clear_output(wait=True)
        show_word_clouds(w_nt_wc.value, w_mode_wc.value)

btn_wc.on_click(on_btn_wc)
display(widgets.HBox([w_nt_wc, w_mode_wc, btn_wc]))
display(out_wc)

### Cell 9: Interactive Topic Explorer

Select an iteration and explore individual topics: their top terms,
document counts, coherence scores, and representative papers.

In [ ]:
# ── Cell 9 — Interactive topic explorer ───────────────────────────────────────

def explore_topics(n_topics):
    """Display detailed topic information for a given n_topics."""
    labels = topic_assignments[n_topics]
    params = iteration_params[n_topics]
    colors = params['colors']
    counts = params['dominant_counts']
    coherences = params['coherence_per_topic']
    exclusivities = params['exclusivities']
    
    # ── Summary table ──
    rows = []
    for t in range(n_topics):
        top5 = ', '.join(top_words_per_iter[n_topics][t][:5])
        rows.append({
            'Topic': t,
            'N_docs': counts[t],
            'Pct': counts[t] / len(labels) * 100,
            'Coherence': coherences[t],
            'Exclusivity': exclusivities[t],
            'Top_5_words': top5
        })
    
    topic_df = pd.DataFrame(rows)
    
    def color_topic_row(row):
        t = int(row['Topic'])
        bg = colors[t]
        r, g, b = int(bg[1:3], 16)/255, int(bg[3:5], 16)/255, int(bg[5:7], 16)/255
        lum = 0.299*r + 0.587*g + 0.114*b
        fg = '#000000' if lum > 0.45 else '#FFFFFF'
        return [f'background-color: {bg}; color: {fg}'] * len(row)
    
    display(HTML(f'<h3>Topics at n_topics={n_topics}</h3>'))
    display(topic_df.style
            .apply(color_topic_row, axis=1)
            .format({'Pct': '{:.1f}%', 'Coherence': '{:.3f}', 'Exclusivity': '{:.3f}'}))
    
    # ── Membership confidence violin plot ──
    membership_probs = params['membership_probs']
    
    fig, ax = plt.subplots(figsize=(12, 4))
    data_violin = [membership_probs[labels == t] for t in range(n_topics)]
    parts = ax.violinplot(data_violin, positions=range(n_topics),
                          showmeans=True, showmedians=True)
    
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
    
    ax.set_xticks(range(n_topics))
    ax.set_xticklabels([f'T{t}' for t in range(n_topics)], fontsize=8)
    ax.set_ylabel('Membership Probability')
    ax.set_title(f'Membership Confidence (n_topics={n_topics})', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


# ── Widget ──
w_nt_explore = widgets.IntSlider(value=15, min=ITER_MIN, max=ITER_MAX, step=1,
                                  description='n_topics:')
btn_explore = widgets.Button(description='Explore', button_style='primary')
out_explore = widgets.Output()

def on_btn_explore(b):
    with out_explore:
        clear_output(wait=True)
        explore_topics(w_nt_explore.value)

btn_explore.on_click(on_btn_explore)
display(widgets.HBox([w_nt_explore, btn_explore]))
display(out_explore)

### Cell 10: Transitions FROM a Specific Topic to a Later Iteration

Given a source topic at iteration K₁ (e.g., n_topics=15, topic 12 = "time"),
track where its documents end up at K₂ (e.g., n_topics=16). This answers:

> *"When we add one more topic, how does this topic split — and what vocabulary
> characterises each destination?"*

**Outputs:**
1. Summary table of destinations
2. Sankey diagram (one source → multiple destinations)
3. Word clouds for each destination sub-group (showing what vocabulary drives the split)
4. Term-weight line chart comparing destination profiles

The paper uses this to identify the "time" topic losing spatio-temporal papers
when moving from 15 to 16 topics (Section 6.3, Iteration 2).

In [ ]:
# ── Cell 10 — Transitions FROM a specific topic ───────────────────────────────

def show_transitions_from_topic(k1, topic_id, k2):
    """Track where documents of topic_id at k1 end up at k2."""
    labels1 = topic_assignments[k1]
    labels2 = topic_assignments[k2]
    colors1 = iteration_params[k1]['colors']
    colors2 = iteration_params[k2]['colors']
    
    source_mask = (labels1 == topic_id)
    n_source = int(source_mask.sum())
    
    if n_source == 0:
        print(f'⚠ No documents in n_topics={k1}, topic {topic_id}')
        return
    
    # Count destinations
    dest_labels = labels2[source_mask]
    dest_topics, counts = np.unique(dest_labels, return_counts=True)
    sort_idx = np.argsort(-counts)
    dest_topics = dest_topics[sort_idx]
    counts = counts[sort_idx]
    pcts = counts / n_source * 100
    
    # Summary
    top_src = ', '.join(top_words_per_iter[k1][topic_id][:5])
    print(f'\n{"═"*65}')
    print(f'  FROM K={k1}.T{topic_id} "{top_src}" ({n_source} docs)')
    print(f'  TO K={k2}')
    print(f'{"═"*65}')
    for dt, cnt, pct in zip(dest_topics, counts, pcts):
        top_d = ', '.join(top_words_per_iter[k2][dt][:3])
        bar = '█' * int(pct / 3)
        print(f'  → K={k2}.T{dt} "{top_d}"  {cnt:>5} ({pct:.1f}%)  {bar}')
    
    # Sankey
    palette = [colors2[int(dt)] for dt in dest_topics]
    n_dest = len(dest_topics)
    node_labels = [f'K={k1}.T{topic_id}<br>{top_src}<br>({n_source})']
    node_colors = [colors1[topic_id]]
    for i, dt in enumerate(dest_topics):
        top_d = ', '.join(top_words_per_iter[k2][dt][:3])
        node_labels.append(f'K={k2}.T{dt}<br>{top_d}<br>({counts[i]})')
        node_colors.append(palette[i])
    
    link_colors = []
    for p in palette:
        r, g, b = int(p[1:3], 16), int(p[3:5], 16), int(p[5:7], 16)
        link_colors.append(f'rgba({r},{g},{b},0.4)')
    
    fig = go.Figure(go.Sankey(
        node=dict(pad=20, thickness=25, line=dict(color='black', width=0.5),
                  label=node_labels, color=node_colors),
        link=dict(source=[0]*n_dest, target=list(range(1, n_dest+1)),
                  value=counts.tolist(), color=link_colors)
    ))
    fig.update_layout(title=f'Transitions: K={k1}.T{topic_id} → K={k2}',
                      height=max(350, 50*n_dest), width=700)
    fig.show()
    
    # ── Word clouds for each destination sub-group ──
    ncols = min(n_dest, 3)
    nrows = math.ceil(n_dest / ncols)
    fig_wc, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.5*nrows))
    if n_dest == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if nrows > 1 else list(axes)
    
    for i, dt in enumerate(dest_topics):
        ax = axes[i]
        sub_mask = source_mask & (labels2 == dt)
        sub_docs = [docs_processed[j] for j in range(len(docs_processed)) if sub_mask[j]]
        
        word_counts = Counter()
        vocab_set = set(terms)
        for doc in sub_docs:
            word_counts.update(w for w in doc.split() if w in vocab_set)
        freq = dict(sorted(word_counts.items(), key=lambda x: -x[1])[:50])
        
        if freq:
            rgb = tuple(int(palette[i][j:j+2], 16) for j in (1, 3, 5))
            wc = WordCloud(width=600, height=400, max_words=50,
                          background_color='white',
                          color_func=lambda w, **kw: f'rgb({rgb[0]},{rgb[1]},{rgb[2]})',
                          collocations=False).generate_from_frequencies(freq)
            ax.imshow(wc, interpolation='bilinear')
        
        top_d = ', '.join(top_words_per_iter[k2][dt][:3])
        ax.set_title(f'→ T{dt}: {top_d} (N={counts[i]})', fontsize=9, fontweight='bold')
        ax.axis('off')
    
    for idx in range(n_dest, len(axes)):
        axes[idx].axis('off')
    
    fig_wc.suptitle(f'Word clouds: K={k1}.T{topic_id} → destinations at K={k2}',
                    fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # ── Term-weight comparison (line chart) ──
    H_source = topic_term_weights[k1][topic_id]
    top_term_idx = H_source.argsort()[-20:][::-1]
    top_term_names = [terms[i] for i in top_term_idx]
    
    fig_line, ax_line = plt.subplots(figsize=(12, 5))
    x_pos = range(len(top_term_names))
    
    ax_line.plot(x_pos, H_source[top_term_idx], 'k-', linewidth=3, alpha=0.4,
                label=f'K={k1}.T{topic_id} (source)', zorder=1)
    
    H2 = topic_term_weights[k2]
    for i, dt in enumerate(dest_topics[:6]):
        ax_line.plot(x_pos, H2[dt][top_term_idx], color=palette[i],
                     linewidth=2, marker='o', markersize=4,
                     label=f'→ K={k2}.T{dt} (N={counts[i]})', zorder=2)
    
    ax_line.set_xticks(x_pos)
    ax_line.set_xticklabels(top_term_names, rotation=45, ha='right', fontsize=8)
    ax_line.set_ylabel('Term weight')
    ax_line.set_title(f'Term-weight profiles: K={k1}.T{topic_id} → K={k2}', fontweight='bold')
    ax_line.legend(fontsize=8)
    ax_line.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


# ── Widget ──
w_k1_from = widgets.IntSlider(value=15, min=ITER_MIN, max=ITER_MAX, step=1,
                               description='Source K:')
w_t_from = widgets.IntText(value=0, description='Topic:')
w_k2_from = widgets.IntSlider(value=16, min=ITER_MIN, max=ITER_MAX, step=1,
                               description='Target K:')
btn_from = widgets.Button(description='Show Transitions', button_style='info')
out_from = widgets.Output()

def on_btn_from(b):
    with out_from:
        clear_output(wait=True)
        k1, t, k2 = w_k1_from.value, w_t_from.value, w_k2_from.value
        if t >= k1:
            print(f'⚠ Topic must be 0..{k1-1}')
            return
        show_transitions_from_topic(k1, t, k2)

btn_from.on_click(on_btn_from)
display(widgets.HBox([w_k1_from, w_t_from, w_k2_from, btn_from]))
display(out_from)

### Cell 11: Transitions TO a Specific Topic from an Earlier Iteration

The reverse of Cell 10: given a target topic at K₂, trace where its documents
came from at K₁. Answers:

> *"Is this topic a clean descendant of one parent, or does it aggregate
> documents from multiple sources (wastebasket pattern)?"*

The paper uses this to identify the "method" wastebasket topic at n_topics=16,
which draws from multiple parents (visual analytics, graphs, rendering, etc.).

In [ ]:
# ── Cell 11 — Transitions TO a specific topic ─────────────────────────────────

def show_transitions_to_topic(k1, k2, topic_id):
    """Trace origins of topic_id at k2 from k1."""
    labels1 = topic_assignments[k1]
    labels2 = topic_assignments[k2]
    colors1 = iteration_params[k1]['colors']
    colors2 = iteration_params[k2]['colors']
    
    target_mask = (labels2 == topic_id)
    n_target = int(target_mask.sum())
    
    if n_target == 0:
        print(f'⚠ No documents in n_topics={k2}, topic {topic_id}')
        return
    
    # Count origins
    origin_labels = labels1[target_mask]
    origin_topics, counts = np.unique(origin_labels, return_counts=True)
    sort_idx = np.argsort(-counts)
    origin_topics = origin_topics[sort_idx]
    counts = counts[sort_idx]
    pcts = counts / n_target * 100
    
    # Summary
    top_tgt = ', '.join(top_words_per_iter[k2][topic_id][:5])
    print(f'\n{"═"*65}')
    print(f'  TO K={k2}.T{topic_id} "{top_tgt}" ({n_target} docs)')
    print(f'  FROM K={k1}')
    print(f'{"═"*65}')
    for ot, cnt, pct in zip(origin_topics, counts, pcts):
        top_o = ', '.join(top_words_per_iter[k1][ot][:3])
        bar = '█' * int(pct / 3)
        print(f'  ← K={k1}.T{ot} "{top_o}"  {cnt:>5} ({pct:.1f}%)  {bar}')
    
    # Detect wastebasket pattern
    if len(origin_topics) >= 4 and pcts[0] < 40:
        print(f'\n  ⚠ WASTEBASKET PATTERN: topic draws from {len(origin_topics)} sources,'
              f' largest only {pcts[0]:.1f}%')
    
    # Sankey
    palette = [colors1[int(ot)] for ot in origin_topics]
    n_orig = len(origin_topics)
    node_labels = []
    node_colors = []
    for i, ot in enumerate(origin_topics):
        top_o = ', '.join(top_words_per_iter[k1][ot][:3])
        node_labels.append(f'K={k1}.T{ot}<br>{top_o}<br>({counts[i]})')
        node_colors.append(palette[i])
    node_labels.append(f'K={k2}.T{topic_id}<br>{top_tgt}<br>({n_target})')
    node_colors.append(colors2[topic_id])
    
    link_colors = []
    for p in palette:
        r, g, b = int(p[1:3], 16), int(p[3:5], 16), int(p[5:7], 16)
        link_colors.append(f'rgba({r},{g},{b},0.4)')
    
    fig = go.Figure(go.Sankey(
        node=dict(pad=20, thickness=25, line=dict(color='black', width=0.5),
                  label=node_labels, color=node_colors),
        link=dict(source=list(range(n_orig)), target=[n_orig]*n_orig,
                  value=counts.tolist(), color=link_colors)
    ))
    fig.update_layout(title=f'Origins: K={k1} → K={k2}.T{topic_id}',
                      height=max(350, 50*n_orig), width=700)
    fig.show()
    
    # ── Word clouds per origin sub-group ──
    ncols = min(n_orig, 3)
    nrows = math.ceil(n_orig / ncols)
    fig_wc, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.5*nrows))
    if n_orig == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if nrows > 1 else list(axes)
    
    vocab_set = set(terms)
    for i, ot in enumerate(origin_topics):
        ax = axes[i]
        sub_mask = target_mask & (labels1 == ot)
        sub_docs = [docs_processed[j] for j in range(len(docs_processed)) if sub_mask[j]]
        
        word_counts = Counter()
        for doc in sub_docs:
            word_counts.update(w for w in doc.split() if w in vocab_set)
        freq = dict(sorted(word_counts.items(), key=lambda x: -x[1])[:50])
        
        if freq:
            rgb = tuple(int(palette[i][j:j+2], 16) for j in (1, 3, 5))
            wc = WordCloud(width=600, height=400, max_words=50,
                          background_color='white',
                          color_func=lambda w, _rgb=rgb, **kw: f'rgb({_rgb[0]},{_rgb[1]},{_rgb[2]})',
                          collocations=False).generate_from_frequencies(freq)
            ax.imshow(wc, interpolation='bilinear')
        
        top_o = ', '.join(top_words_per_iter[k1][ot][:3])
        ax.set_title(f'← T{ot}: {top_o} (N={counts[i]})', fontsize=9, fontweight='bold')
        ax.axis('off')
    
    for idx in range(n_orig, len(axes)):
        axes[idx].axis('off')
    
    fig_wc.suptitle(f'Origins of K={k2}.T{topic_id} from K={k1}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ── Widget ──
w_k1_to = widgets.IntSlider(value=15, min=ITER_MIN, max=ITER_MAX, step=1,
                             description='Source K:')
w_k2_to = widgets.IntSlider(value=16, min=ITER_MIN, max=ITER_MAX, step=1,
                             description='Target K:')
w_t_to = widgets.IntText(value=0, description='Topic:')
btn_to = widgets.Button(description='Show Origins', button_style='warning')
out_to = widgets.Output()

def on_btn_to(b):
    with out_to:
        clear_output(wait=True)
        k1, k2, t = w_k1_to.value, w_k2_to.value, w_t_to.value
        if t >= k2:
            print(f'⚠ Topic must be 0..{k2-1}')
            return
        show_transitions_to_topic(k1, k2, t)

btn_to.on_click(on_btn_to)
display(widgets.HBox([w_k1_to, w_k2_to, w_t_to, btn_to]))
display(out_to)

### Cell 12: Pair Transition Between Two Specific Topics

Three-class decomposition between K₁.T₁ and K₂.T₂:
- 🟢 **Shared (core):** documents in both topics
- 🔴 **Source only (lost):** in T₁ but not T₂
- 🔵 **Target only (gained):** in T₂ but not T₁

Plus word-cloud comparison showing what vocabulary distinguishes each class.
The paper uses this to analyse the 106 spatio-temporal papers lost from the
"time" topic (15.12 → 16.12).

In [ ]:
# ── Cell 12 — Pair transition (three-class decomposition) ─────────────────────

def show_transition_between_topics(k1, t1, k2, t2):
    """Three-class decomposition between two specific topics."""
    labels1 = topic_assignments[k1]
    labels2 = topic_assignments[k2]
    
    mask_t1 = (labels1 == t1)
    mask_t2 = (labels2 == t2)
    
    shared = mask_t1 & mask_t2
    source_only = mask_t1 & ~mask_t2
    target_only = ~mask_t1 & mask_t2
    
    n_t1 = int(mask_t1.sum())
    n_t2 = int(mask_t2.sum())
    n_shared = int(shared.sum())
    n_source_only = int(source_only.sum())
    n_target_only = int(target_only.sum())
    
    if n_t1 == 0 or n_t2 == 0:
        print('⚠ One or both topics are empty')
        return
    
    jaccard = n_shared / (n_t1 + n_t2 - n_shared)
    retention = n_shared / n_t1 * 100
    purity = n_shared / n_t2 * 100
    
    top1 = ', '.join(top_words_per_iter[k1][t1][:5])
    top2 = ', '.join(top_words_per_iter[k2][t2][:5])
    
    print(f'\n{"═"*65}')
    print(f'  K={k1}.T{t1} "{top1}" ({n_t1} docs)')
    print(f'  K={k2}.T{t2} "{top2}" ({n_t2} docs)')
    print(f'{"═"*65}')
    print(f'  Shared (core):    {n_shared:>5}  ({retention:.1f}% of source, {purity:.1f}% of target)')
    print(f'  Source only:      {n_source_only:>5}  ({n_source_only/n_t1*100:.1f}% of source)')
    print(f'  Target only:      {n_target_only:>5}  ({n_target_only/n_t2*100:.1f}% of target)')
    print(f'  Jaccard:          {jaccard:.3f}')
    
    # Bar chart
    fig_bar = go.Figure(go.Bar(
        x=['Source only<br>(lost)', 'Shared<br>(core)', 'Target only<br>(gained)'],
        y=[n_source_only, n_shared, n_target_only],
        marker_color=['#d62728', '#2ca02c', '#1f77b4'],
        text=[f'{n_source_only}', f'{n_shared}', f'{n_target_only}'],
        textposition='outside'
    ))
    fig_bar.update_layout(title=f'K={k1}.T{t1} → K={k2}.T{t2} | Jaccard={jaccard:.3f}',
                          height=350, width=500)
    fig_bar.show()
    
    # ── Word clouds for each class ──
    class_info = [
        ('Shared (core)', shared, n_shared, '#2ca02c'),
        (f'K={k1}.T{t1} only (lost)', source_only, n_source_only, '#d62728'),
        (f'K={k2}.T{t2} only (gained)', target_only, n_target_only, '#1f77b4'),
    ]
    
    fig_wc, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    vocab_set = set(terms)
    
    for i, (label, mask, n, color) in enumerate(class_info):
        ax = axes[i]
        if n == 0:
            ax.text(0.5, 0.5, f'{label}\n(empty)', ha='center', va='center',
                    transform=ax.transAxes)
            ax.axis('off')
            continue
        
        sub_docs = [docs_processed[j] for j in range(len(docs_processed)) if mask[j]]
        word_counts = Counter()
        for doc in sub_docs:
            word_counts.update(w for w in doc.split() if w in vocab_set)
        freq = dict(sorted(word_counts.items(), key=lambda x: -x[1])[:50])
        
        if freq:
            rgb = tuple(int(color[j:j+2], 16) for j in (1, 3, 5))
            wc = WordCloud(width=600, height=400, max_words=50,
                          background_color='white',
                          color_func=lambda w, _rgb=rgb, **kw: f'rgb({_rgb[0]},{_rgb[1]},{_rgb[2]})',
                          collocations=False).generate_from_frequencies(freq)
            ax.imshow(wc, interpolation='bilinear')
        
        ax.set_title(f'{label} (N={n})', fontsize=10, fontweight='bold')
        ax.axis('off')
    
    fig_wc.suptitle(f'Three-class word clouds: K={k1}.T{t1} → K={k2}.T{t2}',
                    fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # ── Term-weight difference: what distinguishes "lost" from "core"? ──
    if n_source_only > 0 and n_shared > 0:
        lost_docs = [docs_processed[j] for j in range(len(docs_processed)) if source_only[j]]
        core_docs = [docs_processed[j] for j in range(len(docs_processed)) if shared[j]]
        
        lost_counts = Counter()
        core_counts = Counter()
        for doc in lost_docs:
            lost_counts.update(w for w in doc.split() if w in vocab_set)
        for doc in core_docs:
            core_counts.update(w for w in doc.split() if w in vocab_set)
        
        # Normalise by document count
        all_words = set(lost_counts.keys()) | set(core_counts.keys())
        diffs = {}
        for w in all_words:
            lost_rate = lost_counts.get(w, 0) / n_source_only
            core_rate = core_counts.get(w, 0) / n_shared
            diffs[w] = lost_rate - core_rate
        
        # Top differentiating terms
        sorted_diffs = sorted(diffs.items(), key=lambda x: -abs(x[1]))[:20]
        
        fig_diff, ax = plt.subplots(figsize=(12, 4))
        words_d = [w for w, _ in sorted_diffs]
        vals_d = [v for _, v in sorted_diffs]
        colors_d = ['#d62728' if v > 0 else '#2ca02c' for v in vals_d]
        ax.barh(range(len(words_d)), vals_d, color=colors_d, alpha=0.8)
        ax.set_yticks(range(len(words_d)))
        ax.set_yticklabels(words_d, fontsize=9)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlabel('Δ rate (lost − core)')
        ax.set_title('Terms distinguishing "lost" documents from "core"\n'
                     '(Red = more frequent in lost; Green = more frequent in core)',
                     fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()


# ── Widget ──
w_k1_pair = widgets.IntSlider(value=15, min=ITER_MIN, max=ITER_MAX, step=1, description='K₁:')
w_t1_pair = widgets.IntText(value=12, description='T₁:')
w_k2_pair = widgets.IntSlider(value=16, min=ITER_MIN, max=ITER_MAX, step=1, description='K₂:')
w_t2_pair = widgets.IntText(value=12, description='T₂:')
btn_pair = widgets.Button(description='Show Transition', button_style='success')
out_pair = widgets.Output()

def on_btn_pair(b):
    with out_pair:
        clear_output(wait=True)
        show_transition_between_topics(w_k1_pair.value, w_t1_pair.value,
                                       w_k2_pair.value, w_t2_pair.value)

btn_pair.on_click(on_btn_pair)
display(widgets.HBox([w_k1_pair, w_t1_pair, widgets.Label(' → '),
                      w_k2_pair, w_t2_pair, btn_pair]))
display(out_pair)

### Cell 13: Temporal Prevalence Analysis

Following Section 6.3 (Iteration 3) of the paper: aggregate papers by dominant topic
and publication year to reveal long-term trends in the VIS research landscape.

Eight visualisations are produced:
- 2 chart types: stacked area vs. line
- 2 quantity types: absolute count vs. proportion per year
- 2 smoothing options: raw vs. Gaussian-smoothed (σ=1.5)

**Key findings from the paper:**
- Visual analytics emerged ~2004, stabilised at ~15% share
- User studies grew from <5% to ~20% (reflecting methodological maturation)
- Volume rendering declined from dominant 1990s shares to <5%
- High-dimensional data analysis remained remarkably stable (~8–10%)

In [ ]:
# ── Cell 13 — Temporal prevalence analysis ────────────────────────────────────

def temporal_prevalence(n_topics_selected=15):
    """Generate 8 temporal prevalence charts for the selected topic solution."""
    labels = topic_assignments[n_topics_selected]
    colors = iteration_params[n_topics_selected]['colors']
    
    # Add topic assignment to dataframe
    df_temp = df[['Year']].copy()
    df_temp['Topic'] = labels
    
    # Count papers per year and topic
    counts = df_temp.groupby(['Year', 'Topic']).size().unstack(fill_value=0)
    counts = counts.sort_index()
    
    # Ensure all topics present
    for t in range(n_topics_selected):
        if t not in counts.columns:
            counts[t] = 0
    counts = counts[list(range(n_topics_selected))]
    
    # Percentages
    pcts = counts.div(counts.sum(axis=1), axis=0) * 100
    
    # Smoothed versions
    counts_smooth = counts.apply(lambda col: gaussian_filter1d(col.values.astype(float), sigma=SIGMA_SMOOTH))
    pcts_smooth = pcts.apply(lambda col: gaussian_filter1d(col.values.astype(float), sigma=SIGMA_SMOOTH))
    
    topics = list(range(n_topics_selected))
    years = counts.index
    
    # Topic labels for legend
    topic_labels = []
    for t in topics:
        top3 = ', '.join(top_words_per_iter[n_topics_selected][t][:3])
        topic_labels.append(f'T{t}: {top3}')
    
    def plot_stacked(ax, data, ylabel, title):
        bottom = np.zeros(len(data))
        for t in topics:
            ax.fill_between(data.index, bottom, bottom + data[t].values,
                           color=colors[t], alpha=0.8, label=topic_labels[t])
            bottom += data[t].values
        ax.set_xlabel('Year')
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight='bold')
    
    def plot_lines(ax, data, ylabel, title):
        for t in topics:
            ax.plot(data.index, data[t], color=colors[t],
                    linewidth=1.5, label=topic_labels[t])
        ax.set_xlabel('Year')
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight='bold')
    
    # ── 8 charts ──
    charts = [
        ('stack', counts, 'Papers', 'Stacked Area — Absolute (Raw)'),
        ('stack', counts_smooth, 'Papers', f'Stacked Area — Absolute (Smoothed σ={SIGMA_SMOOTH})'),
        ('stack', pcts, '% of Papers', 'Stacked Area — Proportion (Raw)'),
        ('stack', pcts_smooth, '% of Papers', f'Stacked Area — Proportion (Smoothed σ={SIGMA_SMOOTH})'),
        ('line', counts, 'Papers', 'Lines — Absolute (Raw)'),
        ('line', counts_smooth, 'Papers', f'Lines — Absolute (Smoothed σ={SIGMA_SMOOTH})'),
        ('line', pcts, '% of Papers', 'Lines — Proportion (Raw)'),
        ('line', pcts_smooth, '% of Papers', f'Lines — Proportion (Smoothed σ={SIGMA_SMOOTH})'),
    ]
    
    for chart_type, data, ylabel, title in charts:
        fig, ax = plt.subplots(figsize=(14, 6))
        full_title = f'{title} ({n_topics_selected} topics)'
        
        if chart_type == 'stack':
            plot_stacked(ax, data, ylabel, full_title)
            if '%' in ylabel:
                ax.set_ylim(0, 100)
        else:
            plot_lines(ax, data, ylabel, full_title)
        
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


# ── Widget ──
w_nt_temporal = widgets.IntSlider(value=15, min=ITER_MIN, max=ITER_MAX, step=1,
                                   description='n_topics:')
btn_temporal = widgets.Button(description='Show Temporal Analysis',
                               button_style='success')
out_temporal = widgets.Output()

def on_btn_temporal(b):
    with out_temporal:
        clear_output(wait=True)
        temporal_prevalence(w_nt_temporal.value)

btn_temporal.on_click(on_btn_temporal)
display(widgets.HBox([w_nt_temporal, btn_temporal]))
display(out_temporal)

### Cell 14: Static HTML Export

Generates static versions of all key visualisations for HTML export.
Auto-selects representative iterations and transitions.

**Included:**
- Quality metrics (6 charts)
- Sankey flow diagram (full range + detail around selected K)
- Word clouds for selected n_topics (both weighting modes)
- Topic explorer (violin plots + summary table)
- Transition FROM (largest topic split)
- Transition TO (wastebasket detection at K+1)
- Pair transition (time topic: K.12 → K+1.12)
- Temporal prevalence (8 charts)

In [ ]:
# ── Cell 14 — Static HTML export ─────────────────────────────────────────────

from datetime import datetime as dt_now

print(f'{"═" * 70}')
print(f'  STATIC EXPORT — NMF Topic Modelling (IEEE VIS Papers)')
print(f'  {dt_now.now().strftime("%Y-%m-%d %H:%M")}')
print(f'{"═" * 70}')

# ── Select reference n_topics ──
# Based on the paper: n_topics = 15 is the recommended solution
SELECTED_K = 15
if SELECTED_K > ITER_MAX:
    SELECTED_K = metrics_df.loc[metrics_df['Coherence_Cv'].idxmax(), 'N_Topics']

EXPORT_KS = sorted(set([ITER_MIN, 10, SELECTED_K, SELECTED_K + 1, ITER_MAX]))
EXPORT_KS = [k for k in EXPORT_KS if ITER_MIN <= k <= ITER_MAX]

print(f'\n  Selected solution: n_topics = {SELECTED_K}')
print(f'  Export iterations: {EXPORT_KS}')

# ═══════════════════════════════════════════════════════════════
#  Part 1: Quality Metrics
# ═══════════════════════════════════════════════════════════════

print(f'\n{"─" * 70}')
print('  Part 1: Quality Metrics')
print(f'{"─" * 70}')

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
for i, (col, title, color, hb) in enumerate(metric_configs):
    ax = axes.flat[i]
    ax.plot(metrics_df['N_Topics'], metrics_df[col], 'o-', color=color, markersize=5)
    ax.set_xlabel('n_topics')
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.axvline(SELECTED_K, color='red', linestyle='--', alpha=0.5, label=f'Selected={SELECTED_K}')
    ax.legend(fontsize=8)
plt.suptitle(f'NMF Quality Metrics', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
#  Part 2: Sankey
# ═══════════════════════════════════════════════════════════════

print(f'\n{"─" * 70}')
print('  Part 2: Sankey Flow')
print(f'{"─" * 70}')

build_topic_sankey()
build_topic_sankey((SELECTED_K - 2, SELECTED_K + 2), max_display=5)

# ═══════════════════════════════════════════════════════════════
#  Part 3: Word Clouds for selected K
# ═══════════════════════════════════════════════════════════════

print(f'\n{"─" * 70}')
print(f'  Part 3: Word Clouds (n_topics={SELECTED_K})')
print(f'{"─" * 70}')

display(HTML(f'<h3>Word Clouds — n_topics={SELECTED_K} (TF-IDF weighted)</h3>'))
show_word_clouds(SELECTED_K, 'term_weight')

display(HTML(f'<h3>Word Clouds — n_topics={SELECTED_K} (Frequency weighted)</h3>'))
show_word_clouds(SELECTED_K, 'frequency')

# ═══════════════════════════════════════════════════════════════
#  Part 4: Topic Explorer
# ═══════════════════════════════════════════════════════════════

print(f'\n{"─" * 70}')
print(f'  Part 4: Topic Explorer (n_topics={SELECTED_K})')
print(f'{"─" * 70}')

explore_topics(SELECTED_K)

# ═══════════════════════════════════════════════════════════════
#  Part 5: Transition Analysis
# ═══════════════════════════════════════════════════════════════

print(f'\n{"═" * 70}')
print('  Part 5: TRANSITION ANALYSIS')
print(f'{"═" * 70}')

K1_TRANS = SELECTED_K
K2_TRANS = SELECTED_K + 1

if K2_TRANS <= ITER_MAX:
    # 5a: FROM — find topic that loses most documents
    labels1 = topic_assignments[K1_TRANS]
    labels2 = topic_assignments[K2_TRANS]
    
    max_loss = 0
    max_loss_topic = 0
    for t in range(K1_TRANS):
        n_at_k1 = (labels1 == t).sum()
        # How many stay in same-numbered topic at k2?
        if t < K2_TRANS:
            n_stay = ((labels1 == t) & (labels2 == t)).sum()
        else:
            n_stay = 0
        loss = n_at_k1 - n_stay
        if loss > max_loss:
            max_loss = loss
            max_loss_topic = t
    
    print(f'\n  5a: FROM K={K1_TRANS}.T{max_loss_topic} (loses {max_loss} docs) → K={K2_TRANS}')
    display(HTML(f'<h3>🔀 Transition FROM: K={K1_TRANS}.T{max_loss_topic} → K={K2_TRANS}</h3>'))
    show_transitions_from_topic(K1_TRANS, max_loss_topic, K2_TRANS)
    
    # 5b: TO — find the new topic at K2 (if any)
    # The new topic is the one with index = K2-1 (last added)
    new_topic = K2_TRANS - 1
    print(f'\n  5b: TO K={K2_TRANS}.T{new_topic} (the new topic) ← K={K1_TRANS}')
    display(HTML(f'<h3>🔀 Transition TO: K={K1_TRANS} → K={K2_TRANS}.T{new_topic}</h3>'))
    show_transitions_to_topic(K1_TRANS, K2_TRANS, new_topic)
    
    # 5c: Pair — best Jaccard match
    best_jaccard = -1
    best_pair = (0, 0)
    for t1 in range(K1_TRANS):
        mask1 = (labels1 == t1)
        n1 = mask1.sum()
        if n1 == 0: continue
        for t2 in range(K2_TRANS):
            mask2 = (labels2 == t2)
            n2 = mask2.sum()
            if n2 == 0: continue
            n_shared = (mask1 & mask2).sum()
            jac = n_shared / (n1 + n2 - n_shared)
            if jac > best_jaccard:
                best_jaccard = jac
                best_pair = (t1, t2)
    
    p_t1, p_t2 = best_pair
    print(f'\n  5c: Pair K={K1_TRANS}.T{p_t1} ↔ K={K2_TRANS}.T{p_t2} (Jaccard={best_jaccard:.3f})')
    display(HTML(f'<h3>🔀 Pair: K={K1_TRANS}.T{p_t1} → K={K2_TRANS}.T{p_t2} '
                 f'(Jaccard={best_jaccard:.3f})</h3>'))
    show_transition_between_topics(K1_TRANS, p_t1, K2_TRANS, p_t2)

# ═══════════════════════════════════════════════════════════════
#  Part 6: Temporal Prevalence
# ═══════════════════════════════════════════════════════════════

print(f'\n{"═" * 70}')
print(f'  Part 6: Temporal Prevalence (n_topics={SELECTED_K})')
print(f'{"═" * 70}')

temporal_prevalence(SELECTED_K)

# ═══════════════════════════════════════════════════════════════
#  Summary
# ═══════════════════════════════════════════════════════════════

print(f'\n{"═" * 70}')
print(f'  STATIC EXPORT COMPLETE')
print(f'{"═" * 70}')
print(f'  Corpus: {len(df):,} papers (1990–2024)')
print(f'  Vocabulary: {len(terms)} terms ({N_UNIGRAMS} unigrams + {N_BIGRAMS} bigrams)')
print(f'  Iterations: n_topics = {ITER_MIN}–{ITER_MAX}')
print(f'  Selected solution: n_topics = {SELECTED_K}')
print(f'  HDBSCAN archetypes: {n_archetypes} ({(1-noise_frac)*100:.0f}% non-noise)')
print(f'\n  To export as HTML:')
print(f'    File → Download as → HTML (.html)')
print(f'    or: jupyter nbconvert --to html --execute NMF-VIS-Papers.ipynb')
print(f'{"═" * 70}')